In [1]:
# ------------------------------------------------------------------
# Import Required Libraries
#
# pathlib : Handles file system paths in a platform-independent way.
# pandas  : Loads and analyzes tabular datasets.
# numpy   : Supports numerical operations used throughout the project.
# ------------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# Configure Project Directories
#
# Define the root project directory and the location of the source
# datasets used throughout the notebook.
# ------------------------------------------------------------------

PROJECT_PATH = Path("..")

DATASET_PATH = PROJECT_PATH / "datasets"

ARTIFACT_PATH = PROJECT_PATH / "artifacts"


# ------------------------------------------------------------------
# Discover Available Datasets
#
# Search recursively for all CSV files available in the datasets
# directory and display the discovered source files.
# ------------------------------------------------------------------

csv_files = sorted(DATASET_PATH.rglob("*.csv"))

print(f"Datasets discovered: {len(csv_files)}\n")

for file in csv_files:
    print(file.relative_to(DATASET_PATH))

Datasets discovered: 9

commerce\raw\olist_order_items_dataset.csv
commerce\raw\olist_order_payments_dataset.csv
commerce\raw\olist_orders_dataset.csv
commerce\raw\olist_products_dataset.csv
commerce\raw\olist_sellers_dataset.csv
commerce\raw\product_category_name_translation.csv
custommer\raw\olist_customers_dataset.csv
custommer\raw\olist_order_reviews_dataset.csv
delivery\raw\olist_geolocation_dataset.csv


In [2]:
# ------------------------------------------------------------------
# Build Data Profiling Report
#
# Analyze the structure and quality of every dataset discovered
# during the inventory phase.
#
# This profiling report will later support data quality checks,
# transformation design, and Lakehouse modeling decisions.
# ------------------------------------------------------------------

profiling = []

for file in csv_files:

    df = pd.read_csv(file)

    for column in df.columns:

        profiling.append({
            "Domain": file.parts[-3].capitalize(),
            "Dataset": file.stem,
            "Column": column,
            "Data Type": str(df[column].dtype),
            "Rows": len(df),
            "Null Count": int(df[column].isna().sum()),
            "Null Percentage (%)": round(
                (df[column].isna().sum() / len(df)) * 100, 2
            ),
            "Unique Values": int(df[column].nunique()),
            "Unique Percentage (%)": round(
                (df[column].nunique() / len(df)) * 100, 2
            )
        })

enterprise_data_profile = pd.DataFrame(profiling)

enterprise_data_profile

,Domain,Dataset,Column,Data Type,Rows,Null Count,Null Percentage (%),Unique Values,Unique Percentage (%)
0,Commerce,olist_order_items_dataset,order_id,str,112650,0,0.00,98666,87.59
1,Commerce,olist_order_items_dataset,order_item_id,int64,112650,0,0.00,21,0.02
2,Commerce,olist_order_items_dataset,product_id,str,112650,0,0.00,32951,29.25
3,Commerce,olist_order_items_dataset,seller_id,str,112650,0,0.00,3095,2.75
4,Commerce,olist_order_items_dataset,shipping_limit_date,str,112650,0,0.00,93318,82.84
5,Commerce,olist_order_items_dataset,price,float64,112650,0,0.00,5968,5.30
6,Commerce,olist_order_items_dataset,freight_value,float64,112650,0,0.00,6999,6.21
7,Commerce,olist_order_payments_dataset,order_id,str,103886,0,0.00,99440,95.72
8,Commerce,olist_order_payments_dataset,payment_sequential,int64,103886,0,0.00,29,0.03
9,Commerce,olist_order_payments_dataset,payment_type,str,103886,0,0.00,5,0.00


In [3]:
# ------------------------------------------------------------------
# Export Enterprise Data Profile
#
# Persist the profiling results as a reusable data asset.
#
# This artifact provides dataset quality metrics, schema information,
# and structural insights for future data quality processes.
# ------------------------------------------------------------------

profiling_path = ARTIFACT_PATH / "profiling"

profiling_path.mkdir(
    parents=True,
    exist_ok=True
)

enterprise_data_profile.to_csv(
    profiling_path / "enterprise_data_profile.csv",
    index=False
)